# 🚀 Neo-Horcrox — Full ML Pipeline

Pipeline lengkap untuk **3 objective** supply chain analytics:
1. **Late Delivery Risk** — klasifikasi binary (CatBoost vs PyCaret, pilih terbaik)
2. **Demand Forecasting** — prediksi sales harian per kategori (LightGBM)
3. **Supplier Selection** — scoring supplier berdasarkan performa historis (XGBoost)

**Output:** semua artifact langsung tersimpan ke `backend/artifacts/` siap deploy.

---
**Cara pakai:** Run All Cells dari atas. Setiap section independen tapi urutan harus dijaga (data → engineering → model).

## 0. Setup & Import

In [ ]:
import numpy as np
import pandas as pd
import json
import pickle
import warnings
import os
from pathlib import Path
from datetime import datetime

# Sklearn
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.pipeline import Pipeline

# Models
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.catboost

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT         = Path('../../')          # project root
DATA_PATH    = ROOT / 'dataset' / 'DataCoSupplyChainDataset.csv'
ARTIFACTS    = ROOT / 'backend' / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print('✅ Setup selesai')
print('Data:', DATA_PATH)
print('Artifacts:', ARTIFACTS.resolve())

---
## 1. Data Ingestion & Inspection

In [ ]:
df_raw = pd.read_csv(DATA_PATH, encoding='latin1')
print(f'Shape: {df_raw.shape}')
print(f'Kolom: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print('\n=== Target Distribution ===')
print(df_raw['Late_delivery_risk'].value_counts())
print(f'Balance ratio: {df_raw["Late_delivery_risk"].mean():.2%}')

print('\n=== Date Range ===')
print('From:', df_raw['order date (DateOrders)'].min())
print('To:  ', df_raw['order date (DateOrders)'].max())

---
## 2. Feature Engineering (shared untuk semua model)

In [ ]:
df = df_raw.copy()

# ── 2.1 Drop kolom PII / tidak relevan ────────────────────────────────────
DROP_COLS = [
    'Product Description', 'Product Image', 'Product Category Id',
    'Order Zipcode', 'Customer Fname', 'Customer Lname', 'Customer Email',
    'Customer Street', 'Customer Zipcode', 'Customer Password',
    'Order Customer Id',
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

# ── 2.2 Handle missing ─────────────────────────────────────────────────────
df.dropna(subset=['Delivery Status'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── 2.3 Parse tanggal ─────────────────────────────────────────────────────
df['order_date']    = pd.to_datetime(df['order date (DateOrders)'],    errors='coerce')
df['shipping_date'] = pd.to_datetime(df['shipping date (DateOrders)'], errors='coerce')

# ── 2.4 Fitur tanggal ─────────────────────────────────────────────────────
df['order_year']       = df['order_date'].dt.year
df['order_month']      = df['order_date'].dt.month
df['order_day']        = df['order_date'].dt.day
df['order_dayofweek']  = df['order_date'].dt.dayofweek
df['order_hour']       = df['order_date'].dt.hour
df['order_is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(int)
df['order_quarter']    = df['order_date'].dt.quarter

# ── 2.5 Fitur shipping / delay ────────────────────────────────────────────
df['actual_delay'] = (
    df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
)
df['shipping_speed_ratio'] = (
    df['Days for shipping (real)'] /
    df['Days for shipment (scheduled)'].replace(0, np.nan)
)
df['is_late']      = (df['actual_delay'] > 0).astype(int)
df['is_early']     = (df['actual_delay'] < 0).astype(int)
df['severe_delay'] = (df['actual_delay'] > 5).astype(int)

# ── 2.6 Konsistensi finansial ─────────────────────────────────────────────
df['calculated_item_total'] = (
    df['Order Item Product Price'] * df['Order Item Quantity']
    - df['Order Item Discount']
)
df['item_total_gap']        = df['Order Item Total'] - df['calculated_item_total']
df['abs_item_total_gap']    = df['item_total_gap'].abs()
df['has_price_inconsistency'] = (df['abs_item_total_gap'] > 1e-3).astype(int)

# ── 2.7 Geo mismatch ──────────────────────────────────────────────────────
df['country_mismatch'] = (df['Customer Country'] != df['Order Country']).astype(int)
df['state_mismatch']   = (df['Customer State']   != df['Order State']).astype(int)
df['city_mismatch']    = (df['Customer City']    != df['Order City']).astype(int)

# ── 2.8 Price ratio ───────────────────────────────────────────────────────
df['Price_Ratio'] = (
    df['Order Item Product Price'] /
    df['Product Price'].replace(0, np.nan)
)

# ── 2.9 Profit margin ─────────────────────────────────────────────────────
df['profit_margin'] = (
    df['Order Profit Per Order'] /
    df['Sales'].replace(0, np.nan)
)

print(f'Shape setelah engineering: {df.shape}')
print(f'Fitur baru: actual_delay, shipping_speed_ratio, has_price_inconsistency, geo_mismatch, Price_Ratio, profit_margin')

---
## 3. MODEL 1 — Late Delivery Risk
Binary classification: `Late_delivery_risk` (0 = on time, 1 = late)

In [ ]:
# ── 3.1 Feature selection untuk risk model ────────────────────────────────
LEAKAGE_COLS = [
    'Late_delivery_risk', 'Delivery Status',
    'actual_delay', 'is_late', 'is_early', 'severe_delay',
    'shipping_speed_ratio',
    'Days for shipping (real)',   # leakage langsung
    'order date (DateOrders)', 'shipping date (DateOrders)',
    'order_date', 'shipping_date',
]

risk_feature_cols = [
    c for c in df.columns
    if c not in LEAKAGE_COLS
    and df[c].dtype != 'datetime64[ns]'
]

X_risk = df[risk_feature_cols].copy()
y_risk = df['Late_delivery_risk'].copy()

print(f'Features untuk risk model: {len(risk_feature_cols)}')
print(risk_feature_cols)

In [ ]:
# ── 3.2 Encoding kategorikal (Frequency Encoding + Rare Category Handling) ─
RARE_THRESHOLD = 20
category_cols  = X_risk.select_dtypes(include='object').columns.tolist()
numeric_cols   = X_risk.select_dtypes(include=['int64','float64']).columns.tolist()

freq_maps = {}   # disimpan ke artifacts

X_risk_enc = X_risk.copy()

for col in category_cols:
    mode_val = X_risk_enc[col].mode()[0]
    X_risk_enc[col] = X_risk_enc[col].fillna(mode_val)

    counts = X_risk_enc[col].value_counts()
    valid_cats = counts[counts >= RARE_THRESHOLD].index.tolist()
    X_risk_enc[col] = X_risk_enc[col].where(X_risk_enc[col].isin(valid_cats), '__rare__')

    freq = X_risk_enc[col].value_counts(normalize=True).to_dict()
    X_risk_enc[col] = X_risk_enc[col].map(freq)

    freq_maps[col] = freq
    freq_maps[col]['__mode__'] = mode_val
    freq_maps[col]['__valid_categories__'] = valid_cats

print(f'Encoding selesai. Kolom kategorikal: {len(category_cols)}')

In [ ]:
# ── 3.3 Scaling numerik ───────────────────────────────────────────────────
scaler = StandardScaler()
X_risk_enc[numeric_cols] = scaler.fit_transform(X_risk_enc[numeric_cols])

# ── 3.4 Train / Valid / Test split (GroupShuffleSplit by Order Id) ─────────
groups = df['Order Id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss.split(X_risk_enc, y_risk, groups))

X_train = X_risk_enc.iloc[train_idx]
y_train = y_risk.iloc[train_idx]
X_temp  = X_risk_enc.iloc[temp_idx]
y_temp  = y_risk.iloc[temp_idx]
g_temp  = groups.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, g_temp))

X_valid = X_temp.iloc[val_idx];  y_valid = y_temp.iloc[val_idx]
X_test  = X_temp.iloc[test_idx]; y_test  = y_temp.iloc[test_idx]

print(f'Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}')

In [ ]:
# ── 3.5a Model A: CatBoost ────────────────────────────────────────────────
mlflow.set_experiment('neo-horcrox-risk')

with mlflow.start_run(run_name='catboost_risk'):
    cb_params = {
        'iterations':        500,
        'learning_rate':     0.05,
        'depth':             6,
        'l2_leaf_reg':       3,
        'eval_metric':       'AUC',
        'random_seed':       42,
        'verbose':           100,
        'early_stopping_rounds': 50,
    }
    catboost_model = CatBoostClassifier(**cb_params)
    catboost_model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
    )

    # Evaluate
    cb_pred  = catboost_model.predict(X_test)
    cb_proba = catboost_model.predict_proba(X_test)[:, 1]
    cb_auc   = roc_auc_score(y_test, cb_proba)
    cb_f1    = f1_score(y_test, cb_pred, average='weighted')

    mlflow.log_params(cb_params)
    mlflow.log_metrics({'auc': cb_auc, 'f1_weighted': cb_f1})
    mlflow.catboost.log_model(catboost_model, 'catboost_risk_model')

    print(f'\n[CatBoost] AUC: {cb_auc:.4f} | F1: {cb_f1:.4f}')
    print(classification_report(y_test, cb_pred, target_names=['On Time','Late']))

In [ ]:
# ── 3.5b Model B: LightGBM (pengganti PyCaret untuk kecepatan) ────────────
with mlflow.start_run(run_name='lightgbm_risk'):
    lgb_params = {
        'n_estimators':   500,
        'learning_rate':  0.05,
        'max_depth':      6,
        'num_leaves':     40,
        'min_child_samples': 20,
        'subsample':      0.8,
        'colsample_bytree': 0.8,
        'reg_alpha':      0.1,
        'reg_lambda':     0.1,
        'random_state':   42,
        'n_jobs':         -1,
        'verbose':        -1,
    }
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
    )

    lgb_pred  = lgb_model.predict(X_test)
    lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
    lgb_auc   = roc_auc_score(y_test, lgb_proba)
    lgb_f1    = f1_score(y_test, lgb_pred, average='weighted')

    mlflow.log_params(lgb_params)
    mlflow.log_metrics({'auc': lgb_auc, 'f1_weighted': lgb_f1})
    mlflow.sklearn.log_model(lgb_model, 'lgbm_risk_model')

    print(f'\n[LightGBM] AUC: {lgb_auc:.4f} | F1: {lgb_f1:.4f}')
    print(classification_report(y_test, lgb_pred, target_names=['On Time','Late']))

In [ ]:
# ── 3.6 Pilih model terbaik berdasarkan AUC ───────────────────────────────
print('\n' + '='*50)
print('📊 PERBANDINGAN MODEL — LATE DELIVERY RISK')
print('='*50)
results = [
    ('CatBoost', catboost_model, cb_auc, cb_f1),
    ('LightGBM', lgb_model,     lgb_auc, lgb_f1),
]
results.sort(key=lambda x: x[2], reverse=True)

for name, model, auc, f1 in results:
    marker = '🏆' if name == results[0][0] else '  '
    print(f'{marker} {name:<12} AUC={auc:.4f}  F1={f1:.4f}')

best_risk_name,  best_risk_model, best_risk_auc, best_risk_f1 = results[0]
print(f'\n✅ Model terpilih: {best_risk_name} (AUC={best_risk_auc:.4f})')

In [ ]:
# ── 3.7 Feature importance ────────────────────────────────────────────────
import matplotlib.pyplot as plt

if best_risk_name == 'CatBoost':
    importances = best_risk_model.get_feature_importance()
    feat_names  = X_train.columns
else:
    importances = best_risk_model.feature_importances_
    feat_names  = X_train.columns

feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh')
plt.title(f'Top 20 Feature Importance — {best_risk_name}')
plt.tight_layout()
plt.show()

print('\nTop 10 Features:')
print(feat_imp.head(10))

In [ ]:
# ── 3.8 Export artifacts Risk Model ───────────────────────────────────────
# Model
with open(ARTIFACTS / 'risk_model.pkl', 'wb') as f:
    pickle.dump(best_risk_model, f)

# Scaler
with open(ARTIFACTS / 'risk_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Freq maps (untuk encoding di inference)
with open(ARTIFACTS / 'risk_freq_maps.json', 'w') as f:
    json.dump(freq_maps, f, indent=2)

# Feature list (urutan kolom saat training)
with open(ARTIFACTS / 'risk_features.json', 'w') as f:
    json.dump(list(X_train.columns), f)

# Metadata
risk_meta = {
    'model_name':  best_risk_name,
    'auc':         round(best_risk_auc, 4),
    'f1_weighted': round(best_risk_f1, 4),
    'n_features':  len(risk_feature_cols),
    'train_size':  len(X_train),
    'test_size':   len(X_test),
    'trained_at':  datetime.now().isoformat(),
}
with open(ARTIFACTS / 'risk_metadata.json', 'w') as f:
    json.dump(risk_meta, f, indent=2)

print('✅ Risk model artifacts tersimpan:')
for p in sorted(ARTIFACTS.glob('risk_*')):
    print(f'   {p.name} ({p.stat().st_size / 1024:.1f} KB)')

---
## 4. MODEL 2 — Demand Forecasting
Prediksi `Sales` harian per `Category Name` × `Market` menggunakan LightGBM dengan lag features.

In [ ]:
# ── 4.1 Aggregate sales harian ────────────────────────────────────────────
df_ts = df[['order_date', 'Category Name', 'Market', 'Sales', 'Order Item Quantity']].copy()
df_ts['date'] = df_ts['order_date'].dt.date

daily = (
    df_ts
    .groupby(['date', 'Category Name', 'Market'])
    .agg(
        total_sales=('Sales', 'sum'),
        total_qty=('Order Item Quantity', 'sum'),
        order_count=('Sales', 'count')
    )
    .reset_index()
)
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values(['Category Name', 'Market', 'date']).reset_index(drop=True)

print(f'Daily aggregat shape: {daily.shape}')
print(f'Date range: {daily["date"].min()} → {daily["date"].max()}')
daily.head()

In [ ]:
# ── 4.2 Feature engineering untuk forecasting ─────────────────────────────
def make_forecast_features(df_g: pd.DataFrame) -> pd.DataFrame:
    """Buat lag + rolling features per group (Category × Market)."""
    df_g = df_g.copy().sort_values('date')

    # Kalender
    df_g['dayofweek']  = df_g['date'].dt.dayofweek
    df_g['month']      = df_g['date'].dt.month
    df_g['quarter']    = df_g['date'].dt.quarter
    df_g['is_weekend'] = df_g['dayofweek'].isin([5, 6]).astype(int)
    df_g['year']       = df_g['date'].dt.year

    # Lag features
    for lag in [1, 7, 14, 30]:
        df_g[f'lag_{lag}'] = df_g['total_sales'].shift(lag)

    # Rolling features
    for win in [7, 14, 30]:
        df_g[f'roll_mean_{win}'] = df_g['total_sales'].shift(1).rolling(win).mean()
        df_g[f'roll_std_{win}']  = df_g['total_sales'].shift(1).rolling(win).std()
        df_g[f'roll_max_{win}']  = df_g['total_sales'].shift(1).rolling(win).max()

    return df_g

daily_feat = (
    daily
    .groupby(['Category Name', 'Market'], group_keys=False)
    .apply(make_forecast_features)
    .dropna()
    .reset_index(drop=True)
)

print(f'Shape setelah lag features: {daily_feat.shape}')
print(f'Columns: {daily_feat.columns.tolist()}')

In [ ]:
# ── 4.3 Encode Category dan Market ────────────────────────────────────────
fc_cat_le = LabelEncoder()
fc_mkt_le = LabelEncoder()

daily_feat['cat_encoded'] = fc_cat_le.fit_transform(daily_feat['Category Name'])
daily_feat['mkt_encoded'] = fc_mkt_le.fit_transform(daily_feat['Market'])

# ── 4.4 Feature list & split (time-based) ─────────────────────────────────
FC_FEATURES = [
    'cat_encoded', 'mkt_encoded',
    'dayofweek', 'month', 'quarter', 'is_weekend', 'year',
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_30',
    'roll_std_7', 'roll_std_14', 'roll_std_30',
    'roll_max_7', 'roll_max_14', 'roll_max_30',
    'order_count', 'total_qty',
]
TARGET_FC = 'total_sales'

# 80/20 time-based split
cutoff = daily_feat['date'].quantile(0.80)
train_fc = daily_feat[daily_feat['date'] <= cutoff]
test_fc  = daily_feat[daily_feat['date']  > cutoff]

X_fc_train, y_fc_train = train_fc[FC_FEATURES], train_fc[TARGET_FC]
X_fc_test,  y_fc_test  = test_fc[FC_FEATURES],  test_fc[TARGET_FC]

print(f'Train: {X_fc_train.shape} | Test: {X_fc_test.shape}')
print(f'Cutoff date: {cutoff.date()}')

In [ ]:
# ── 4.5 Training LightGBM Regressor ───────────────────────────────────────
mlflow.set_experiment('neo-horcrox-forecast')

with mlflow.start_run(run_name='lgbm_forecast'):
    fc_params = {
        'n_estimators':      1000,
        'learning_rate':     0.03,
        'max_depth':         6,
        'num_leaves':        40,
        'min_child_samples': 10,
        'subsample':         0.8,
        'colsample_bytree':  0.8,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
    }
    forecast_model = lgb.LGBMRegressor(**fc_params)
    forecast_model.fit(
        X_fc_train, y_fc_train,
        eval_set=[(X_fc_test, y_fc_test)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
    )

    fc_pred = forecast_model.predict(X_fc_test)
    fc_mae  = mean_absolute_error(y_fc_test, fc_pred)
    fc_rmse = np.sqrt(mean_squared_error(y_fc_test, fc_pred))
    fc_r2   = r2_score(y_fc_test, fc_pred)

    mlflow.log_params(fc_params)
    mlflow.log_metrics({'mae': fc_mae, 'rmse': fc_rmse, 'r2': fc_r2})
    mlflow.sklearn.log_model(forecast_model, 'forecast_model')

    print(f'[Forecast LightGBM] MAE={fc_mae:.2f} | RMSE={fc_rmse:.2f} | R²={fc_r2:.4f}')

In [ ]:
# ── 4.6 Export Forecast artifacts ─────────────────────────────────────────
with open(ARTIFACTS / 'forecast_model.pkl', 'wb') as f:
    pickle.dump(forecast_model, f)

with open(ARTIFACTS / 'forecast_cat_encoder.pkl', 'wb') as f:
    pickle.dump(fc_cat_le, f)

with open(ARTIFACTS / 'forecast_mkt_encoder.pkl', 'wb') as f:
    pickle.dump(fc_mkt_le, f)

# Simpan statistik per group untuk generate forecast
group_stats = (
    daily
    .groupby(['Category Name', 'Market'])
    .agg(
        mean_sales=('total_sales', 'mean'),
        std_sales=('total_sales', 'std'),
        mean_qty=('total_qty', 'mean'),
        last_date=('date', 'max')
    )
    .reset_index()
)
group_stats['last_date'] = group_stats['last_date'].astype(str)
group_stats.to_json(ARTIFACTS / 'forecast_group_stats.json', orient='records', indent=2)

forecast_meta = {
    'features':    FC_FEATURES,
    'target':      TARGET_FC,
    'mae':         round(fc_mae, 4),
    'rmse':        round(fc_rmse, 4),
    'r2':          round(fc_r2, 4),
    'cutoff_date': str(cutoff.date()),
    'categories':  fc_cat_le.classes_.tolist(),
    'markets':     fc_mkt_le.classes_.tolist(),
    'trained_at':  datetime.now().isoformat(),
}
with open(ARTIFACTS / 'forecast_metadata.json', 'w') as f:
    json.dump(forecast_meta, f, indent=2)

print('✅ Forecast artifacts tersimpan:')
for p in sorted(ARTIFACTS.glob('forecast_*')):
    print(f'   {p.name} ({p.stat().st_size / 1024:.1f} KB)')

---
## 5. MODEL 3 — Supplier Selection
Scoring supplier (Department) berdasarkan: delivery performance, profit ratio, dan volume.
Output: skor 0–1 per (Department × Shipping Mode × Market) + XGBoost regressor untuk scoring baru.

In [ ]:
# ── 5.1 Buat Supplier Performance Table ───────────────────────────────────
supplier_raw = df[[
    'Department Name', 'Category Name', 'Market',
    'Shipping Mode', 'Order Region',
    'Late_delivery_risk', 'actual_delay',
    'Order Profit Per Order', 'Order Item Profit Ratio',
    'Sales', 'Order Item Quantity', 'Order Status',
    'has_price_inconsistency',
]].copy()

# Aggregate per supplier (Department + Shipping Mode + Market)
supplier_agg = (
    supplier_raw
    .groupby(['Department Name', 'Shipping Mode', 'Market'])
    .agg(
        total_orders=('Late_delivery_risk', 'count'),
        late_rate=('Late_delivery_risk', 'mean'),           # rendah = lebih baik
        mean_delay=('actual_delay', 'mean'),                # rendah = lebih baik
        std_delay=('actual_delay', 'std'),                  # rendah = lebih konsisten
        mean_profit=('Order Profit Per Order', 'mean'),     # tinggi = lebih baik
        mean_profit_ratio=('Order Item Profit Ratio', 'mean'),
        total_sales=('Sales', 'sum'),
        mean_qty=('Order Item Quantity', 'mean'),
        price_inconsistency_rate=('has_price_inconsistency', 'mean'), # rendah = lebih baik
    )
    .reset_index()
    .dropna()
)

print(f'Supplier aggregated table: {supplier_agg.shape}')
supplier_agg.head()

In [ ]:
# ── 5.2 Hitung composite supplier score (0–1, higher = better) ────────────
from sklearn.preprocessing import MinMaxScaler

supp_scaler = MinMaxScaler()

# Normalize metrik (invert yang 'lebih rendah = lebih baik')
supplier_agg['score_delivery']    = 1 - supplier_agg['late_rate']
supplier_agg['score_delay']       = 1 - supp_scaler.fit_transform(
    supplier_agg[['mean_delay']].fillna(0)
).flatten()
supplier_agg['score_consistency'] = 1 - supp_scaler.fit_transform(
    supplier_agg[['std_delay']].fillna(0)
).flatten()
supplier_agg['score_profit']      = supp_scaler.fit_transform(
    supplier_agg[['mean_profit']].fillna(0)
).flatten()
supplier_agg['score_integrity']   = 1 - supplier_agg['price_inconsistency_rate']

# Composite score — bobot bisa disesuaikan
WEIGHTS = {
    'score_delivery':    0.35,
    'score_delay':       0.25,
    'score_consistency': 0.15,
    'score_profit':      0.15,
    'score_integrity':   0.10,
}
supplier_agg['supplier_score'] = sum(
    supplier_agg[col] * w for col, w in WEIGHTS.items()
)

print('Score distribution:')
print(supplier_agg['supplier_score'].describe())
print('\nTop 10 suppliers:')
print(
    supplier_agg
    .sort_values('supplier_score', ascending=False)
    [['Department Name','Shipping Mode','Market','supplier_score','late_rate','total_orders']]
    .head(10)
)

In [ ]:
# ── 5.3 Train XGBoost untuk prediksi skor supplier baru ───────────────────
# Encode kategoricals
sup_dept_le = LabelEncoder()
sup_mode_le = LabelEncoder()
sup_mkt_le  = LabelEncoder()

supplier_agg['dept_enc'] = sup_dept_le.fit_transform(supplier_agg['Department Name'])
supplier_agg['mode_enc'] = sup_mode_le.fit_transform(supplier_agg['Shipping Mode'])
supplier_agg['mkt_enc']  = sup_mkt_le.fit_transform(supplier_agg['Market'])

SUP_FEATURES = [
    'dept_enc', 'mode_enc', 'mkt_enc',
    'total_orders', 'late_rate', 'mean_delay', 'std_delay',
    'mean_profit', 'mean_profit_ratio', 'total_sales',
    'mean_qty', 'price_inconsistency_rate',
]

X_sup = supplier_agg[SUP_FEATURES].fillna(0)
y_sup = supplier_agg['supplier_score']

from sklearn.model_selection import train_test_split
X_sup_tr, X_sup_te, y_sup_tr, y_sup_te = train_test_split(
    X_sup, y_sup, test_size=0.2, random_state=42
)

mlflow.set_experiment('neo-horcrox-supplier')

with mlflow.start_run(run_name='xgb_supplier'):
    sup_params = {
        'n_estimators':     300,
        'learning_rate':    0.05,
        'max_depth':        4,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'random_state':     42,
        'n_jobs':           -1,
    }
    supplier_model = xgb.XGBRegressor(**sup_params)
    supplier_model.fit(
        X_sup_tr, y_sup_tr,
        eval_set=[(X_sup_te, y_sup_te)],
        verbose=50,
    )

    sup_pred = supplier_model.predict(X_sup_te)
    sup_mae  = mean_absolute_error(y_sup_te, sup_pred)
    sup_r2   = r2_score(y_sup_te, sup_pred)

    mlflow.log_params(sup_params)
    mlflow.log_metrics({'mae': sup_mae, 'r2': sup_r2})
    mlflow.sklearn.log_model(supplier_model, 'supplier_model')

    print(f'[Supplier XGBoost] MAE={sup_mae:.4f} | R²={sup_r2:.4f}')

In [ ]:
# ── 5.4 Export Supplier artifacts ─────────────────────────────────────────
with open(ARTIFACTS / 'supplier_model.pkl', 'wb') as f:
    pickle.dump(supplier_model, f)

with open(ARTIFACTS / 'supplier_dept_encoder.pkl', 'wb') as f:
    pickle.dump(sup_dept_le, f)

with open(ARTIFACTS / 'supplier_mode_encoder.pkl', 'wb') as f:
    pickle.dump(sup_mode_le, f)

with open(ARTIFACTS / 'supplier_mkt_encoder.pkl', 'wb') as f:
    pickle.dump(sup_mkt_le, f)

# Simpan lookup table supplier score historis
supplier_lookup = supplier_agg[[
    'Department Name','Shipping Mode','Market','supplier_score',
    'late_rate','mean_delay','total_orders'
]].copy()
supplier_lookup['supplier_score'] = supplier_lookup['supplier_score'].round(4)
supplier_lookup.to_json(ARTIFACTS / 'supplier_lookup.json', orient='records', indent=2)

supplier_meta = {
    'features':     SUP_FEATURES,
    'target':       'supplier_score',
    'score_weights': WEIGHTS,
    'mae':          round(sup_mae, 4),
    'r2':           round(sup_r2, 4),
    'departments':  sup_dept_le.classes_.tolist(),
    'shipping_modes': sup_mode_le.classes_.tolist(),
    'markets':      sup_mkt_le.classes_.tolist(),
    'trained_at':   datetime.now().isoformat(),
}
with open(ARTIFACTS / 'supplier_metadata.json', 'w') as f:
    json.dump(supplier_meta, f, indent=2)

print('✅ Supplier artifacts tersimpan:')
for p in sorted(ARTIFACTS.glob('supplier_*')):
    print(f'   {p.name} ({p.stat().st_size / 1024:.1f} KB)')

---
## 6. Summary — Semua Artifacts

In [ ]:
print('=' * 60)
print('📦 SEMUA ARTIFACTS TERSIMPAN DI backend/artifacts/')
print('=' * 60)

total_size = 0
for p in sorted(ARTIFACTS.iterdir()):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        total_size += size_kb
        prefix = '🔵' if 'risk' in p.name else ('🟢' if 'forecast' in p.name else '🟡')
        print(f'  {prefix} {p.name:<45} {size_kb:>8.1f} KB')

print(f'\n  Total: {total_size / 1024:.2f} MB')

print('\n📊 RINGKASAN PERFORMA MODEL')
print('-' * 40)
print(f'  🔵 Risk ({best_risk_name})')
print(f'     AUC      = {best_risk_auc:.4f}')
print(f'     F1       = {best_risk_f1:.4f}')
print(f'  🟢 Forecast (LightGBM)')
print(f'     MAE      = {fc_mae:.2f}')
print(f'     RMSE     = {fc_rmse:.2f}')
print(f'     R²       = {fc_r2:.4f}')
print(f'  🟡 Supplier (XGBoost)')
print(f'     MAE      = {sup_mae:.4f}')
print(f'     R²       = {sup_r2:.4f}')

print('\n✅ Pipeline selesai. Backend siap di-start:')
print('   docker compose up --build')